# Health Analysis – Deriving Heart Rate and Breathing Rate from IMU Data

This notebook loads raw IMU (accelerometer + gyroscope) data recorded with the
Health Monitor Flutter app and derives:

1. **Breathing rate** (typical range: 12–20 breaths/min) from low-frequency
   accelerometer oscillations.
2. **Heart rate** (typical range: 50–120 bpm) from higher-frequency
   oscillations caused by the heartbeat impulse (ballistocardiography).

### Method overview
When the phone lies flat on the chest:
- The **Z-axis accelerometer** (perpendicular to chest) captures the dominant
  chest-wall motion.
- **Breathing** causes slow, large-amplitude oscillations (~0.2–0.5 Hz).
- **Heartbeat** causes rapid, small-amplitude impulses (~1–2 Hz).

Signal processing steps:
1. Re-sample to uniform 100 Hz grid.
2. Remove DC offset and long-term drift (detrend).
3. **Breathing**: bandpass filter 0.1–0.6 Hz → find peaks → compute rate.
4. **Heart rate**: bandpass filter 0.8–3 Hz → find peaks → compute rate.
5. Plot time-domain traces and PSD for each.

In [ ]:
# Install dependencies (uncomment if running in a fresh Colab environment)
# !pip install numpy pandas scipy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from scipy.signal import butter, filtfilt, find_peaks, welch
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. Load IMU data

Expected CSV columns: `timestamp_ms, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`

Set `CSV_FILES` to point at your recorded trial files.

In [ ]:
# ------------------------------------------------------------------
# CONFIGURE: paths to your trial CSV files
# ------------------------------------------------------------------
CSV_FILES = [
    'imu_trial_1.csv',
    'imu_trial_2.csv',
    'imu_trial_3.csv',
]

TARGET_FS = 100.0   # Hz – resample target
TRIAL_DURATION_S = 300  # 5 minutes


def load_trial(path: str) -> pd.DataFrame:
    """Load a single trial CSV, validate columns, and sort by timestamp."""
    df = pd.read_csv(path)
    required = ['timestamp_ms', 'acc_x', 'acc_y', 'acc_z',
                 'gyro_x', 'gyro_y', 'gyro_z']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'{path}: missing columns {missing}')
    df = df.sort_values('timestamp_ms').reset_index(drop=True)
    print(f'Loaded {path}: {len(df):,} samples, '
          f'duration = {df["timestamp_ms"].iloc[-1]/1000:.1f} s, '
          f'avg fs ≈ {1000*len(df)/df["timestamp_ms"].iloc[-1]:.1f} Hz')
    return df


trials_raw = []
for path in CSV_FILES:
    if Path(path).exists():
        trials_raw.append(load_trial(path))
    else:
        print(f'⚠ File not found: {path} – skipping')

print(f'\nLoaded {len(trials_raw)} trial(s).')

## 2. Pre-processing helpers

In [ ]:
def resample_uniform(df: pd.DataFrame, fs: float = TARGET_FS) -> np.ndarray:
    """
    Resample all IMU channels to a uniform grid at `fs` Hz using linear
    interpolation.  Returns (t, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z)
    as a dict of 1-D numpy arrays.
    """
    t_orig = df['timestamp_ms'].values / 1000.0  # seconds
    t_new = np.arange(t_orig[0], t_orig[-1], 1.0 / fs)

    channels = {}
    for col in ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']:
        channels[col] = np.interp(t_new, t_orig, df[col].values)

    channels['t'] = t_new
    return channels


def bandpass(data: np.ndarray, low: float, high: float,
             fs: float = TARGET_FS, order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass filter."""
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, data)


def compute_rate_from_peaks(filtered: np.ndarray,
                             fs: float = TARGET_FS,
                             min_hz: float = 0.1,
                             max_hz: float = 3.5) -> tuple[float, np.ndarray]:
    """
    Detect peaks in `filtered` and return (mean_rate_in_bpm_or_bpm,
    peak_indices).
    min_hz / max_hz sets the expected inter-peak spacing.
    """
    min_dist = int(fs / max_hz)  # minimum samples between peaks
    peaks, _ = find_peaks(filtered, distance=min_dist,
                          height=np.std(filtered) * 0.3)
    if len(peaks) < 2:
        return float('nan'), peaks

    # Instantaneous rate from inter-peak intervals
    intervals_s = np.diff(peaks) / fs
    # Filter by plausible interval range
    valid = intervals_s[(intervals_s >= 1/max_hz) & (intervals_s <= 1/min_hz)]
    if len(valid) == 0:
        return float('nan'), peaks

    mean_rate = 60.0 / np.mean(valid)
    return mean_rate, peaks


def dominant_frequency(data: np.ndarray, fs: float = TARGET_FS,
                        fmin: float = 0.1, fmax: float = 3.5) -> float:
    """Return the dominant frequency (Hz) in the band [fmin, fmax] via Welch PSD."""
    freqs, psd = welch(data, fs=fs, nperseg=min(len(data), int(fs * 30)))
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return float('nan')
    return freqs[mask][np.argmax(psd[mask])]

## 3. Analyse each trial

In [ ]:
# Frequency bands (Hz)
BREATHING_LOW, BREATHING_HIGH = 0.1, 0.6
HEART_LOW, HEART_HIGH = 0.8, 3.0

results = []

for idx, df in enumerate(trials_raw):
    trial_num = idx + 1
    ch = resample_uniform(df)
    t = ch['t']

    # Use Z-axis (phone lies flat → Z perpendicular to chest wall)
    acc_z = ch['acc_z'] - np.mean(ch['acc_z'])  # remove DC (gravity component)

    # --- Breathing ---
    breathing_signal = bandpass(acc_z, BREATHING_LOW, BREATHING_HIGH)
    breathing_rate, b_peaks = compute_rate_from_peaks(
        breathing_signal, min_hz=BREATHING_LOW, max_hz=BREATHING_HIGH)
    breathing_freq_psd = dominant_frequency(
        acc_z, fmin=BREATHING_LOW, fmax=BREATHING_HIGH)
    breathing_rate_psd = breathing_freq_psd * 60

    # --- Heart rate ---
    heart_signal = bandpass(acc_z, HEART_LOW, HEART_HIGH)
    heart_rate, h_peaks = compute_rate_from_peaks(
        heart_signal, min_hz=HEART_LOW, max_hz=HEART_HIGH)
    heart_freq_psd = dominant_frequency(
        acc_z, fmin=HEART_LOW, fmax=HEART_HIGH)
    heart_rate_psd = heart_freq_psd * 60

    results.append({
        'trial': trial_num,
        'n_samples': len(t),
        'duration_s': t[-1] - t[0],
        'breathing_rate_peaks': breathing_rate,
        'breathing_rate_psd': breathing_rate_psd,
        'heart_rate_peaks': heart_rate,
        'heart_rate_psd': heart_rate_psd,
        # Store signals for plotting
        '_t': t,
        '_acc_z': acc_z,
        '_breathing': breathing_signal,
        '_b_peaks': b_peaks,
        '_heart': heart_signal,
        '_h_peaks': h_peaks,
    })

    print(f'Trial {trial_num}:')
    print(f'  Breathing rate (peaks): {breathing_rate:.1f} breaths/min')
    print(f'  Breathing rate (PSD):   {breathing_rate_psd:.1f} breaths/min')
    print(f'  Heart rate    (peaks):  {heart_rate:.1f} bpm')
    print(f'  Heart rate    (PSD):    {heart_rate_psd:.1f} bpm')
    print()

## 4. Plot: time-domain traces

In [ ]:
for r in results:
    t = r['_t']
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    fig.suptitle(f'Trial {r["trial"]} – Time-Domain Signals', fontsize=14)

    # Breathing
    ax = axes[0]
    ax.plot(t, r['_breathing'], color='steelblue', linewidth=0.8,
            label='Filtered (0.1–0.6 Hz)')
    ax.plot(t[r['_b_peaks']], r['_breathing'][r['_b_peaks']],
            'rv', markersize=6, label=f'Peaks ({r["breathing_rate_peaks"]:.1f} br/min)')
    ax.set_ylabel('Acc Z (m/s²)')
    ax.set_title('Breathing Signal')
    ax.legend(loc='upper right')

    # Heart rate
    ax = axes[1]
    ax.plot(t, r['_heart'], color='tomato', linewidth=0.6,
            label='Filtered (0.8–3.0 Hz)')
    ax.plot(t[r['_h_peaks']], r['_heart'][r['_h_peaks']],
            'b^', markersize=5, label=f'Peaks ({r["heart_rate_peaks"]:.1f} bpm)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Acc Z (m/s²)')
    ax.set_title('Heart Rate Signal (BCG)')
    ax.legend(loc='upper right')

    plt.tight_layout()
    plt.savefig(f'trial_{r["trial"]}_timeseries.png', bbox_inches='tight')
    plt.show()

## 5. Plot: Power Spectral Density

In [ ]:
for r in results:
    acc_z = r['_acc_z']
    freqs, psd = welch(acc_z, fs=TARGET_FS,
                       nperseg=min(len(acc_z), int(TARGET_FS * 30)))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(freqs, psd, color='navy', linewidth=0.9)

    # Shade breathing band
    mask_br = (freqs >= BREATHING_LOW) & (freqs <= BREATHING_HIGH)
    ax.fill_between(freqs[mask_br], psd[mask_br], alpha=0.3,
                    color='steelblue', label='Breathing band')

    # Shade heart rate band
    mask_hr = (freqs >= HEART_LOW) & (freqs <= HEART_HIGH)
    ax.fill_between(freqs[mask_hr], psd[mask_hr], alpha=0.3,
                    color='tomato', label='Heart rate band')

    # Mark dominant frequencies
    br_f = r['breathing_rate_psd'] / 60
    hr_f = r['heart_rate_psd'] / 60
    ax.axvline(br_f, color='steelblue', linestyle='--',
               label=f'Breathing peak: {br_f:.3f} Hz ({r["breathing_rate_psd"]:.1f} br/min)')
    ax.axvline(hr_f, color='tomato', linestyle='--',
               label=f'Heart rate peak: {hr_f:.3f} Hz ({r["heart_rate_psd"]:.1f} bpm)')

    ax.set_xlim(0, 4)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD [(m/s²)²/Hz]')
    ax.set_title(f'Trial {r["trial"]} – Power Spectral Density (Z-axis)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'trial_{r["trial"]}_psd.png', bbox_inches='tight')
    plt.show()

## 6. Summary table

In [ ]:
summary = pd.DataFrame([
    {
        'Trial': r['trial'],
        'Duration (s)': f"{r['duration_s']:.1f}",
        'Samples': r['n_samples'],
        'Breathing (peaks, br/min)': f"{r['breathing_rate_peaks']:.1f}",
        'Breathing (PSD, br/min)': f"{r['breathing_rate_psd']:.1f}",
        'Heart Rate (peaks, bpm)': f"{r['heart_rate_peaks']:.1f}",
        'Heart Rate (PSD, bpm)': f"{r['heart_rate_psd']:.1f}",
    }
    for r in results
])

print(summary.to_string(index=False))

# Aggregate (mean across trials)
if len(results) > 0:
    print('\n--- Mean across all trials ---')
    print(f'Breathing rate: '
          f"{np.nanmean([r['breathing_rate_psd'] for r in results]):.1f} br/min ")
    print(f'Heart rate:     '
          f"{np.nanmean([r['heart_rate_psd'] for r in results]):.1f} bpm")

## 7. (Bonus) Microphone-based analysis

If you also recorded microphone audio, the cell below attempts to estimate
heart rate and breathing rate from the M4A file.  It requires `librosa`.

**Approach**: Same bandpass → peak-detection strategy applied to the audio
RMS envelope computed in short (50 ms) frames.

In [ ]:
# Uncomment and install if you want to run the bonus analysis:
# !pip install librosa soundfile

AUDIO_FILES = [
    'audio_trial_1.m4a',
    'audio_trial_2.m4a',
    'audio_trial_3.m4a',
]

try:
    import librosa

    for audio_path in AUDIO_FILES:
        if not Path(audio_path).exists():
            print(f'⚠ Audio file not found: {audio_path}')
            continue

        y, sr = librosa.load(audio_path, sr=None, mono=True)
        print(f'{audio_path}: {len(y)/sr:.1f} s @ {sr} Hz')

        # Compute RMS envelope in 50-ms frames
        frame_len = int(sr * 0.05)
        hop_len = frame_len // 2
        rms = librosa.feature.rms(y=y, frame_length=frame_len,
                                   hop_length=hop_len)[0]
        fs_env = sr / hop_len  # frames per second
        t_env = np.arange(len(rms)) / fs_env

        # Breathing
        br_sig = bandpass(rms, BREATHING_LOW, BREATHING_HIGH, fs=fs_env)
        br_rate_audio, _ = compute_rate_from_peaks(
            br_sig, fs=fs_env,
            min_hz=BREATHING_LOW, max_hz=BREATHING_HIGH)

        # Heart rate
        hr_sig = bandpass(rms, HEART_LOW, HEART_HIGH, fs=fs_env)
        hr_rate_audio, _ = compute_rate_from_peaks(
            hr_sig, fs=fs_env,
            min_hz=HEART_LOW, max_hz=HEART_HIGH)

        print(f'  Breathing rate (audio): {br_rate_audio:.1f} br/min')
        print(f'  Heart rate    (audio): {hr_rate_audio:.1f} bpm\n')

        # Quick plot
        fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
        fig.suptitle(f'{audio_path} – Audio Envelope Analysis')
        axes[0].plot(t_env, br_sig, color='steelblue')
        axes[0].set_title(f'Breathing ({br_rate_audio:.1f} br/min)')
        axes[1].plot(t_env, hr_sig, color='tomato')
        axes[1].set_title(f'Heart Rate ({hr_rate_audio:.1f} bpm)')
        axes[1].set_xlabel('Time (s)')
        plt.tight_layout()
        plt.show()

except ImportError:
    print('librosa not installed. Run: pip install librosa soundfile')